In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    USER_DAY_FILE,
    USER_BASELINE_FILE,
    INSIDERS_CSV,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    return Path(path).resolve().as_posix()


# ============================================================
# INPUT CHECK
# ============================================================

required_files = {
    "User-Day dataset": USER_DAY_FILE,
    "Ground truth": INSIDERS_CSV,
}

missing = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing)
    )

print("✓ Required input files found.")


# ============================================================
# DETERMINE BASELINE CUTOFF
# ============================================================

insiders = pd.read_csv(
    INSIDERS_CSV,
    dtype=str,
)

insiders["dataset"] = pd.to_numeric(
    insiders["dataset"],
    errors="coerce",
)

insiders["start"] = pd.to_datetime(
    insiders["start"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)

r42 = insiders[
    insiders["dataset"] == 4.2
].copy()

baseline_cutoff = (
    r42["start"]
    .min()
    .date()
)

print("\n" + "=" * 70)
print("BUILDING USER-SPECIFIC BASELINE")
print("=" * 70)

print(
    f"\nBaseline cutoff : "
    f"{baseline_cutoff}"
)

print(
    "Reference rule  : "
    "weekdays only + active workdays only"
)


# ============================================================
# PATHS
# ============================================================

input_path = db_path(
    USER_DAY_FILE
)

output_path = db_path(
    USER_BASELINE_FILE
)


# ============================================================
# BUILD BASELINE
# ============================================================

con = duckdb.connect()

con.execute(f"""
    COPY (

        WITH reference_period AS (

            SELECT
                *,

                (
                    EXTRACT(
                        HOUR FROM first_logon
                    ) * 60
                    +
                    EXTRACT(
                        MINUTE FROM first_logon
                    )
                ) AS first_logon_minute,

                (
                    EXTRACT(
                        HOUR FROM last_logoff
                    ) * 60
                    +
                    EXTRACT(
                        MINUTE FROM last_logoff
                    )
                ) AS last_logoff_minute

            FROM read_parquet(
                '{input_path}'
            )

            WHERE
                day < DATE '{baseline_cutoff}'

                -- Weekdays only
                AND is_weekend = 0

                -- User was active that day
                AND logon_count > 0
        )

        SELECT
            user,

            COUNT(*) AS baseline_workdays,


            -- ===============================================
            -- EMAIL
            -- ===============================================

            median(
                email_count
            ) AS email_count_median,

            quantile_cont(
                email_count,
                0.95
            ) AS email_count_p95,


            median(
                external_email_events
            ) AS external_email_median,

            quantile_cont(
                external_email_events,
                0.95
            ) AS external_email_p95,


            median(
                total_email_bytes
            ) AS email_bytes_median,

            quantile_cont(
                total_email_bytes,
                0.95
            ) AS email_bytes_p95,


            median(
                attachment_count
            ) AS attachment_median,

            quantile_cont(
                attachment_count,
                0.95
            ) AS attachment_p95,


            -- ===============================================
            -- USB
            -- ===============================================

            median(
                usb_connect_count
            ) AS usb_connect_median,

            quantile_cont(
                usb_connect_count,
                0.95
            ) AS usb_connect_p95,


            median(
                files_copied_to_usb
            ) AS usb_files_median,

            quantile_cont(
                files_copied_to_usb,
                0.95
            ) AS usb_files_p95,


            -- ===============================================
            -- HTTP
            -- ===============================================

            median(
                http_request_count
            ) AS http_requests_median,

            quantile_cont(
                http_request_count,
                0.95
            ) AS http_requests_p95,


            median(
                unique_domains
            ) AS unique_domains_median,

            quantile_cont(
                unique_domains,
                0.95
            ) AS unique_domains_p95,


            -- ===============================================
            -- LOGON BEHAVIOR
            -- ===============================================

            median(
                first_logon_minute
            ) AS first_logon_median,

            quantile_cont(
                first_logon_minute,
                0.05
            ) AS first_logon_p05,

            quantile_cont(
                first_logon_minute,
                0.95
            ) AS first_logon_p95,


            median(
                last_logoff_minute
            ) AS last_logoff_median,

            quantile_cont(
                last_logoff_minute,
                0.05
            ) AS last_logoff_p05,

            quantile_cont(
                last_logoff_minute,
                0.95
            ) AS last_logoff_p95


        FROM reference_period

        GROUP BY user
        ORDER BY user

    )
    TO '{output_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        OVERWRITE_OR_IGNORE TRUE
    )
""")


# ============================================================
# VALIDATION
# ============================================================

summary = con.execute(f"""
    SELECT
        COUNT(*) AS users,
        MIN(baseline_workdays) AS min_workdays,
        MEDIAN(
            baseline_workdays
        ) AS median_workdays,
        MAX(baseline_workdays) AS max_workdays
    FROM read_parquet(
        '{output_path}'
    )
""").df()


duplicates = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            user,
            COUNT(*) AS n
        FROM read_parquet(
            '{output_path}'
        )
        GROUP BY user
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]


critical_nulls = con.execute(f"""
    SELECT COUNT(*)

    FROM read_parquet(
        '{output_path}'
    )

    WHERE
        user IS NULL
        OR baseline_workdays IS NULL
        OR external_email_p95 IS NULL
        OR usb_files_p95 IS NULL
        OR http_requests_p95 IS NULL
""").fetchone()[0]


known_user = con.execute(f"""
    SELECT
        user,
        baseline_workdays,
        external_email_p95,
        usb_files_p95,
        http_requests_p95

    FROM read_parquet(
        '{output_path}'
    )

    WHERE user = 'AAF0535'
""").df()


# ============================================================
# AUTOMATED ASSERTIONS
# ============================================================

assert int(
    summary.iloc[0]["users"]
) == 1000

assert int(
    summary.iloc[0]["min_workdays"]
) >= 20

assert duplicates == 0

assert critical_nulls == 0

assert len(known_user) == 1

assert int(
    known_user.iloc[0][
        "baseline_workdays"
    ]
) == 111

assert float(
    known_user.iloc[0][
        "external_email_p95"
    ]
) == 3.0

assert float(
    known_user.iloc[0][
        "usb_files_p95"
    ]
) == 5.0

assert float(
    known_user.iloc[0][
        "http_requests_p95"
    ]
) == 29.0


# ============================================================
# FINAL STATUS
# ============================================================

print("\nBaseline summary:")
display(summary)

print(
    f"\nDuplicate users  : "
    f"{duplicates}"
)

print(
    f"Critical nulls   : "
    f"{critical_nulls}"
)

print("\nKnown user baseline: AAF0535")
display(known_user)

print("\n" + "=" * 70)
print("✅ BUILD USER BASELINE: PASS")
print("All validation checks passed.")
print("=" * 70)

print(
    f"\nSaved to:\n"
    f"{USER_BASELINE_FILE}"
)

con.close()

✓ Required input files found.

BUILDING USER-SPECIFIC BASELINE

Baseline cutoff : 2010-06-10
Reference rule  : weekdays only + active workdays only

Baseline summary:


,users,min_workdays,median_workdays,max_workdays
0,1000,23,111.0,113



Duplicate users  : 0
Critical nulls   : 0

Known user baseline: AAF0535


,user,baseline_workdays,external_email_p95,usb_files_p95,http_requests_p95
0,AAF0535,111,3.0,5.0,29.0



✅ BUILD USER BASELINE: PASS
All validation checks passed.

Saved to:
C:\Users\HP\Documents\FYP\Data_engineering\data\processed\user_baseline_v1.parquet


In [3]:
# ============================================================
# USER BASELINE INSPECTION
# ============================================================

path = USER_BASELINE_FILE.resolve().as_posix()

con = duckdb.connect()

print("=" * 70)
print("USER BASELINE INSPECTION")
print("=" * 70)


# ============================================================
# BASELINE SUMMARY
# ============================================================

summary = con.execute(f"""
    SELECT
        COUNT(*) AS users,
        MIN(
            baseline_workdays
        ) AS min_days,

        MEDIAN(
            baseline_workdays
        ) AS median_days,

        MAX(
            baseline_workdays
        ) AS max_days

    FROM read_parquet(
        '{path}'
    )
""").df()

print("\nBaseline summary:")
display(summary)


# ============================================================
# KNOWN CERT USER
# ============================================================

aaf_baseline = con.execute(f"""
    SELECT *
    FROM read_parquet(
        '{path}'
    )
    WHERE user = 'AAF0535'
""").df()

con.close()


print("\nAAF0535 baseline:")

display(
    aaf_baseline.T.rename(
        columns={0: "value"}
    )
)

print(
    "\n✅ BASELINE INSPECTION COMPLETE"
)

USER BASELINE INSPECTION

Baseline summary:


,users,min_days,median_days,max_days
0,1000,23,111.0,113



AAF0535 baseline:


,value
user,AAF0535
baseline_workdays,111
email_count_median,3.0
email_count_p95,3.0
external_email_median,1.0
external_email_p95,3.0
email_bytes_median,87999.0
email_bytes_p95,119169.0
attachment_median,2.0
attachment_p95,7.0



✅ BASELINE INSPECTION COMPLETE
